# Dados de Assinatura, Habilitação e Troncos da Telefonia Fixa — DSAC

Versão PySpark da query SQL que consolida, por **serviço/produto Anatel**:

1. **Base e Instalações NET FONE + CLARO FONE** (Residencial x Empresa)
2. **Base e Instalações CLARO FONE** (PUCs Local / Outros Serviços)
3. **Base e Ativações de troncos E1 empresariais** (`dsac_sumexec`)

Tabelas de origem no **AWS Athena**, database/owner **`trusted_db`**:
`trusted_db.dsac_net_fone`, `trusted_db.dsac_claro_fone` e `trusted_db.dsac_sumexec`.

> **Como o Spark lê as tabelas do Athena:** o Athena não guarda dados — as tabelas ficam no
> **AWS Glue Data Catalog** e os arquivos no **S3**. O Spark usa o mesmo Glue Data Catalog como
> metastore e lê os arquivos direto do S3 (sem passar pelo motor do Athena).
> - **Glue Job / Glue Studio Notebook:** habilite *"Use Glue Data Catalog as the Hive metastore"* (`--enable-glue-datacatalog`).
> - **EMR / EMR Studio:** classificação `spark-hive-site` com `hive.metastore.client.factory.class = com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory`.
> - **Role IAM:** `glue:GetDatabase/GetTable/GetPartitions` no `trusted_db` e `s3:GetObject/ListBucket` nos buckets das tabelas
>   (para gravar: `glue:CreateTable/UpdateTable` e `s3:PutObject/DeleteObject` no destino).

A lógica é a mesma do SQL original, inclusive:
- `dsac_claro_fone` é contabilizada nos blocos 1 **e** 2 (como no SQL);
- no bloco 3 o arredondamento é feito **por mês** (`ano_mes`) antes da soma anual.

## 1. Parâmetros
Ajuste aqui o ano de referência, o database das tabelas de origem e o destino da gravação.

In [ ]:
ANO = "2025"                      # ano de referência (4 dígitos)
DATABASE = "trusted_db"           # database (owner) das tabelas no Athena / Glue Data Catalog

TABELA_NET_FONE = "dsac_net_fone"
TABELA_CLARO_FONE = "dsac_claro_fone"
TABELA_SUMEXEC = "dsac_sumexec"

# Destino (opcional). Parquet no S3 + tabela no Glue = consultável no Athena.
SAIDA_TABELA = None               # ex.: "refined_db.dsac_telefonia_fixa"; None = não registra tabela
SAIDA_S3 = None                   # ex.: "s3://meu-bucket/refined/dsac_telefonia_fixa/"; None = não grava
MODO = "overwrite"                # overwrite | append | errorifexists

assert len(ANO) == 4 and ANO.isdigit(), "ANO deve ter 4 dígitos, ex.: 2025"

## 2. Imports e SparkSession
`getOrCreate()` reaproveita a sessão já existente (Glue Studio, EMR Studio, SageMaker). A sessão usa o **Glue Data Catalog** (catálogo do Athena) como metastore.

In [ ]:
import logging
from typing import Iterable, Optional

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s - %(message)s")
log = logging.getLogger("dsac_telefonia_fixa")

spark = (
    SparkSession.builder.appName(f"dsac_telefonia_fixa_{ANO}")
    # Glue Data Catalog (catálogo do Athena) como metastore do Spark
    .config("spark.hadoop.hive.metastore.client.factory.class",
            "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.sql.adaptive.enabled", "true")   # AQE ajusta partições de shuffle
    .config("spark.sql.session.timeZone", "UTC")     # evita diferenças entre ambientes
    # overwrite substitui só as partições gravadas (não apaga outros anos)
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .enableHiveSupport()
    .getOrCreate()
)

## 3. Constantes de negócio
Centralizadas para facilitar a manutenção do de/para Anatel.

In [ ]:
IND_BASE = "Base de Conectados"
IND_INSTALACAO = "Instalação"
INDICADORES_FONE = [IND_BASE, IND_INSTALACAO]

TIPO_RESIDENCIAL = "RESIDENCIAL"
TIPO_EMPRESA = "EMPRESA"

PRODUTO_CLARO_FONE = "Claro Fone"

IND_E1_BASE = "BASE"
IND_E1_ATIVACOES = "ATIVAÇÕES"

NAO_DEFINIDO = "ND"

# Colunas mínimas que cada tabela de origem precisa ter
COLUNAS_FONE = ["data_base", "nm_tipo_produto_marca_bi", "nm_indicador_negocio_bi",
                "nm_linha_negocio_bi", "qtde"]
COLUNAS_SUMEXEC = ["ano_mes", "categoria", "qtd_base", "qtd_ativ"]

# Layout intermediário comum aos 3 blocos (colunas do UNION ALL) e layout final
COLUNAS_UNIAO = ["ano", "produto", "indicador", "servico_anatel", "produto_anatel", "total"]
COLUNAS_SAIDA = ["servico_anatel", "produto_anatel", "total"]


class ErroValidacao(Exception):
    """Erro de qualidade/estrutura dos dados de entrada."""

## 4. Funções auxiliares

In [ ]:
def _ano_de(coluna: str) -> Column:
    """Equivalente a substr(cast(coluna as varchar), 1, 4)."""
    return F.substring(F.col(coluna).cast("string"), 1, 4)


def _filtro_ano(coluna: str, ano: str) -> Column:
    """Equivalente a cast(coluna as varchar) like '<ano>%'."""
    return F.col(coluna).cast("string").startswith(ano)


def _mapear(regras: Iterable[tuple], padrao: str = NAO_DEFINIDO) -> Column:
    """Monta um CASE WHEN ... THEN ... ELSE padrao END a partir de uma lista (condição, valor)."""
    expr = None
    for condicao, valor in regras:
        expr = F.when(condicao, valor) if expr is None else expr.when(condicao, valor)
    return expr.otherwise(padrao)


def validar_colunas(df: DataFrame, obrigatorias: list, nome: str) -> None:
    """Falha cedo (antes de qualquer processamento) se faltar coluna na origem."""
    faltando = [c for c in obrigatorias if c not in df.columns]
    if faltando:
        raise ErroValidacao(f"Tabela '{nome}' sem as colunas obrigatórias: {faltando}")


def ler_tabela_athena(database: str, tabela: str) -> DataFrame:
    """Lê uma tabela do Athena (Glue Data Catalog), validando antes se ela existe.
    O nome vai em minúsculas, pois o Athena/Glue armazena nomes em lowercase."""
    nome = f"{database}.{tabela}".lower()
    if not spark.catalog.tableExists(nome):
        raise ErroValidacao(
            f"Tabela '{nome}' não encontrada no Glue Data Catalog. Verifique se a sessão usa o "
            "Glue como metastore e se a role IAM tem permissão no database."
        )
    log.info("Lendo %s", nome)
    return spark.table(nome)

## 5. Leitura e validação das tabelas de origem (Athena — `trusted_db`)
Só as colunas necessárias são selecionadas (*column pruning* em Parquet/ORC no S3).

In [ ]:
df_net = ler_tabela_athena(DATABASE, TABELA_NET_FONE)
df_claro = ler_tabela_athena(DATABASE, TABELA_CLARO_FONE)
df_sumexec = ler_tabela_athena(DATABASE, TABELA_SUMEXEC)

validar_colunas(df_net, COLUNAS_FONE, TABELA_NET_FONE)
validar_colunas(df_claro, COLUNAS_FONE, TABELA_CLARO_FONE)
validar_colunas(df_sumexec, COLUNAS_SUMEXEC, TABELA_SUMEXEC)

df_net = df_net.select(*COLUNAS_FONE)
df_claro = df_claro.select(*COLUNAS_FONE)
df_sumexec = df_sumexec.select(*COLUNAS_SUMEXEC)

## 6. Bloco 1 — Base e Instalações NET FONE / CLARO FONE
Subquery `t1`: agrupa cada tabela por ano, produto, indicador e linha de negócio, classificando a linha em
**EMPRESA** (contém `'Empresa'`, case-sensitive como o `LIKE`) ou **RESIDENCIAL** (demais casos, inclusive nulos).
Depois aplica o de/para Residencial x Não Residencial.

In [ ]:
def _agregar_fone_por_linha(df: DataFrame, ano: str) -> DataFrame:
    return (
        df.where(_filtro_ano("data_base", ano))
        .where(F.col("nm_indicador_negocio_bi").isin(INDICADORES_FONE))
        .groupBy(
            _ano_de("data_base").alias("ano"),
            F.col("nm_tipo_produto_marca_bi").alias("produto"),
            F.col("nm_indicador_negocio_bi").alias("indicador"),
            F.col("nm_linha_negocio_bi"),
        )
        .agg(F.sum("qtde").alias("total"))
        .withColumn(
            "tipo_produto",
            F.when(F.col("nm_linha_negocio_bi").contains("Empresa"), TIPO_EMPRESA)
            .otherwise(TIPO_RESIDENCIAL),
        )
        .drop("nm_linha_negocio_bi")
    )


def bloco_net_claro_fone(df_net: DataFrame, df_claro: DataFrame, ano: str) -> DataFrame:
    t1 = _agregar_fone_por_linha(df_net, ano).unionByName(_agregar_fone_por_linha(df_claro, ano))

    ind, tipo = F.col("indicador"), F.col("tipo_produto")
    base_res = (ind == IND_BASE) & (tipo == TIPO_RESIDENCIAL)
    base_emp = (ind == IND_BASE) & (tipo == TIPO_EMPRESA)
    inst_res = (ind == IND_INSTALACAO) & (tipo == TIPO_RESIDENCIAL)
    inst_emp = (ind == IND_INSTALACAO) & (tipo == TIPO_EMPRESA)

    return (
        t1.groupBy("ano", "produto", "indicador", "tipo_produto")
        .agg(F.sum("total").alias("total"))
        .withColumn("servico_anatel", _mapear([
            (base_res, "NVTF.1.3.1.1"),
            (base_emp, "NVTF.3.2.1.1"),
            (inst_res, "NVTF.1.3.2.1"),
            (inst_emp, "NVTF.3.2.2.1"),
        ]))
        .withColumn("produto_anatel", _mapear([
            (base_res, "Assinatura Residencial"),
            (base_emp, "Assinatura Não Residencial"),
            (inst_res, "Habilitação Residencial"),
            (inst_emp, "Habilitação Não Residencial"),
        ]))
        .select(*COLUNAS_UNIAO)
    )


df_bloco1 = bloco_net_claro_fone(df_net, df_claro, ANO)

## 7. Bloco 2 — Base e Instalações CLARO FONE
Produtos diferentes de `'Claro Fone'` caem em `'ND'`, como no SQL.

In [ ]:
def bloco_claro_fone(df_claro: DataFrame, ano: str) -> DataFrame:
    ind, prod = F.col("indicador"), F.col("produto")
    base_claro = (ind == IND_BASE) & (prod == PRODUTO_CLARO_FONE)
    inst_claro = (ind == IND_INSTALACAO) & (prod == PRODUTO_CLARO_FONE)

    return (
        df_claro.where(_filtro_ano("data_base", ano))
        .where(F.col("nm_indicador_negocio_bi").isin(INDICADORES_FONE))
        .groupBy(
            _ano_de("data_base").alias("ano"),
            F.col("nm_tipo_produto_marca_bi").alias("produto"),
            F.col("nm_indicador_negocio_bi").alias("indicador"),
        )
        .agg(F.sum("qtde").alias("total"))
        .withColumn("servico_anatel", _mapear([
            (base_claro, "NVTF.8.2.1.1"),
            (inst_claro, "NVTF.8.3.1.1"),
        ]))
        .withColumn("produto_anatel", _mapear([
            (base_claro, "PUCs Local"),
            (inst_claro, "Outras linhas de produto_Outros Serviços_Outros Serviços_Outros Serviços"),
        ]))
        .select(*COLUNAS_UNIAO)
    )


df_bloco2 = bloco_claro_fone(df_claro, ANO)

## 8. Bloco 3 — Base e Ativações E1 Empresarial
Igual ao SQL: soma e **arredonda por mês** (`group by ano_mes, categoria`) e só depois soma no ano.
Base (`qtd_base`) e ativações (`qtd_ativ`) são calculadas na mesma leitura e "despivotadas" com `stack()`,
evitando ler a tabela duas vezes como no `UNION ALL` original.

In [ ]:
def bloco_e1_empresarial(df_sumexec: DataFrame, ano: str) -> DataFrame:
    mensal = (
        df_sumexec.where(F.col("categoria").contains("E1"))
        .where(_filtro_ano("ano_mes", ano))
        .groupBy("ano_mes", "categoria")
        .agg(
            # F.round = HALF_UP (mesmo comportamento do ROUND do Presto/Trino/Hive)
            F.round(F.sum("qtd_base"), 0).alias("tot_base"),
            F.round(F.sum("qtd_ativ"), 0).alias("tot_ativ"),
        )
        .select(
            _ano_de("ano_mes").alias("ano"),
            F.col("categoria").alias("produto"),
            F.expr(f"stack(2, '{IND_E1_BASE}', tot_base, '{IND_E1_ATIVACOES}', tot_ativ) "
                   "as (indicador, total)"),
        )
    )

    ind = F.col("indicador")
    return (
        mensal.groupBy("ano", "produto", "indicador")
        .agg(F.sum("total").alias("total"))
        .withColumn("servico_anatel", _mapear([
            (ind == IND_E1_BASE, "NVTF.2.2.1.1"),
            (ind == IND_E1_ATIVACOES, "NVTF.2.2.2.1"),
        ]))
        .withColumn("produto_anatel", _mapear([
            (ind == IND_E1_BASE, "Assinatura Tronco"),
            (ind == IND_E1_ATIVACOES, "Habilitação Tronco"),
        ]))
        .select(*COLUNAS_UNIAO)
    )


df_bloco3 = bloco_e1_empresarial(df_sumexec, ANO)

## 9. Consolidação final
Une os 3 blocos (`t3`) e totaliza por serviço/produto Anatel (SELECT externo).

In [ ]:
t3 = df_bloco1.unionByName(df_bloco2).unionByName(df_bloco3)

resultado = (
    t3.groupBy("servico_anatel", "produto_anatel")
    .agg(F.sum("total").alias("total"))
    .orderBy("servico_anatel")
    .select(*COLUNAS_SAIDA)
    .cache()   # resultado pequeno, reutilizado na validação, exibição e gravação
)

## 10. Validações do resultado

In [ ]:
qtd = resultado.count()
if qtd == 0:
    raise ErroValidacao(f"Nenhum registro encontrado para o ano {ANO}")

nd = resultado.where(F.col("servico_anatel") == NAO_DEFINIDO).count()
if nd:
    # Mesma regra do SQL (cai em 'ND'), mas avisamos para não passar despercebido
    log.warning("%d linha(s) classificadas como 'ND' (sem de/para Anatel)", nd)

log.info("Resultado com %d linha(s)", qtd)
resultado.show(truncate=False)

## 11. Gravação no S3 / Glue Catalog
Parquet particionado por `ano`, registrado como tabela externa no Glue (consultável no Athena). Reprocessar um ano substitui só a partição daquele ano. O resultado tem poucas linhas, por isso `coalesce(1)` gera um único arquivo.

In [ ]:
saida = resultado.withColumn("ano", F.lit(ANO)).coalesce(1)
escritor = saida.write.mode(MODO).format("parquet").partitionBy("ano")

if SAIDA_TABELA:
    if not SAIDA_S3:
        raise ErroValidacao("SAIDA_TABELA exige SAIDA_S3 (tabela externa para o Athena)")
    log.info("Gravando tabela %s em %s (modo=%s)", SAIDA_TABELA, SAIDA_S3, MODO)
    if MODO == "overwrite" and spark.catalog.tableExists(SAIDA_TABELA):
        # Tabela já existe: insertInto respeita o overwrite dinâmico por partição.
        # insertInto é posicional, então alinhamos as colunas na ordem da tabela.
        colunas_destino = spark.table(SAIDA_TABELA).columns
        saida.select(*colunas_destino).write.insertInto(SAIDA_TABELA, overwrite=True)
    else:
        escritor.option("path", SAIDA_S3).saveAsTable(SAIDA_TABELA)
elif SAIDA_S3:
    log.info("Gravando parquet em %s (modo=%s)", SAIDA_S3, MODO)
    escritor.save(SAIDA_S3)

resultado.unpersist()
log.info("Processo finalizado com sucesso")